# Clean End-to-End Hypertension Pipeline
This notebook is intentionally structured as one clear step per cell: reading, merging, target creation, cleaning, imputation, training, calibration, explainability, and export.

## 1) Imports and global config

In [46]:
import os
import json
import warnings
import pickle
import numpy as np
import pandas as pd
import optuna
from datetime import datetime
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid, ParameterSampler
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, roc_auc_score, log_loss
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.inspection import permutation_importance
from sklearn.ensemble import StackingClassifier
from sklearn.base import clone
from sklearn.feature_selection import mutual_info_classif

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

ROOT = Path.cwd()
print('Working directory:', ROOT)

# ---------------- checkpointing (crash-safe) ----------------
CKPT_DIR = ROOT / '_checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def _atomic_pickle_dump(obj, path: Path):
    tmp = path.with_suffix(path.suffix + '.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    os.replace(tmp, path)

def save_ckpt(name: str, obj):
    _atomic_pickle_dump(obj, CKPT_DIR / f'{name}.pkl')

def load_ckpt(name: str, default=None):
    p = CKPT_DIR / f'{name}.pkl'
    if p.exists():
        with open(p, 'rb') as f:
            return pickle.load(f)
    return default

def save_df_progress(df: pd.DataFrame, filename: str):
    p = CKPT_DIR / filename
    tmp = p.with_suffix(p.suffix + '.tmp')
    df.to_csv(tmp, index=False)
    os.replace(tmp, p)

def mark_stage(stage: str):
    with open(CKPT_DIR / 'last_stage.txt', 'w', encoding='utf-8') as f:
        f.write(f"{datetime.now().isoformat()} | {stage}")

print('Checkpoint directory:', CKPT_DIR)

# ---------------- GPU detection (safe fallback to CPU) ----------------
USE_GPU = True

def _detect_xgb_gpu():
    if not USE_GPU:
        return False
    try:
        X_tmp = np.array([[0.0], [1.0], [2.0], [3.0]])
        y_tmp = np.array([0, 0, 1, 1])
        m = XGBClassifier(
            n_estimators=1, max_depth=1, learning_rate=0.1,
            eval_metric='logloss', random_state=RANDOM_SEED,
            device='cuda', tree_method='hist', verbosity=0
        )
        m.fit(X_tmp, y_tmp)
        return True
    except Exception:
        return False

def _detect_lgbm_gpu():
    if not USE_GPU:
        return False
    try:
        X_tmp = np.array([[0.0], [1.0], [2.0], [3.0]])
        y_tmp = np.array([0, 0, 1, 1])
        m = LGBMClassifier(
            n_estimators=1, num_leaves=7, random_state=RANDOM_SEED,
            device_type='gpu', verbosity=-1
        )
        m.fit(X_tmp, y_tmp)
        return True
    except Exception:
        return False

def _detect_cat_gpu():
    if not USE_GPU:
        return False
    try:
        X_tmp = np.array([[0.0], [1.0], [2.0], [3.0]])
        y_tmp = np.array([0, 0, 1, 1])
        m = CatBoostClassifier(
            iterations=1, depth=2, learning_rate=0.1,
            random_seed=RANDOM_SEED, verbose=0, task_type='GPU'
        )
        m.fit(X_tmp, y_tmp)
        return True
    except Exception:
        return False

XGB_GPU_AVAILABLE = _detect_xgb_gpu()
LGBM_GPU_AVAILABLE = _detect_lgbm_gpu()
CAT_GPU_AVAILABLE = _detect_cat_gpu()

print(f'XGBoost GPU available: {XGB_GPU_AVAILABLE}')
print(f'LightGBM GPU available: {LGBM_GPU_AVAILABLE}')
print(f'CatBoost GPU available: {CAT_GPU_AVAILABLE}')

Working directory: c:\Jon\College\Thesis\V2.2.1.1
Checkpoint directory: c:\Jon\College\Thesis\V2.2.1.1\_checkpoints
XGBoost GPU available: True
LightGBM GPU available: True
CatBoost GPU available: True


## 2) File paths

In [3]:
paths = {
    'clinical': ROOT / 'Datasets' / 'Clinical' / 'Jonathan Ralph_Baes_2025-09-18020631_data-set_clinical.csv',
    'anthro': ROOT / 'Datasets' / 'Anthro' / 'Jonathan Ralph_Baes_2025-09-29103911_data-set_anthrop.csv',
    'dietary': ROOT / 'Datasets' / 'Dietary' / 'Jonathan Ralph Florentino Baes_2026-02-08065501_data-set_dietary_indiv.csv',
    'biochem': ROOT / 'Datasets' / 'Biochemical' / 'Jonathan Ralph_Baes_2025-09-29103839_data-set_biochemical.csv',
    'socio': ROOT / 'Datasets' / 'Socio Economic' / 'Jonathan Ralph_Baes_2025-09-29104255_data-set_socio.csv',
}

for k, p in paths.items():
    print(f'{k:8s} | exists={p.exists()} | {p}')

clinical | exists=True | c:\Jon\College\Thesis\V2.2.1.1\Datasets\Clinical\Jonathan Ralph_Baes_2025-09-18020631_data-set_clinical.csv
anthro   | exists=True | c:\Jon\College\Thesis\V2.2.1.1\Datasets\Anthro\Jonathan Ralph_Baes_2025-09-29103911_data-set_anthrop.csv
dietary  | exists=True | c:\Jon\College\Thesis\V2.2.1.1\Datasets\Dietary\Jonathan Ralph Florentino Baes_2026-02-08065501_data-set_dietary_indiv.csv
biochem  | exists=True | c:\Jon\College\Thesis\V2.2.1.1\Datasets\Biochemical\Jonathan Ralph_Baes_2025-09-29103839_data-set_biochemical.csv
socio    | exists=True | c:\Jon\College\Thesis\V2.2.1.1\Datasets\Socio Economic\Jonathan Ralph_Baes_2025-09-29104255_data-set_socio.csv


## 3) Read datasets

In [4]:
raw_data_clinical = pd.read_csv(paths['clinical'], low_memory=False)
raw_data_anthro = pd.read_csv(paths['anthro'], low_memory=False)
raw_data_dietary = pd.read_csv(paths['dietary'], low_memory=False)
raw_data_biochemical = pd.read_csv(paths['biochem'], low_memory=False)
raw_data_socio = pd.read_csv(paths['socio'], low_memory=False)

for name, df in {
    'clinical': raw_data_clinical,
    'anthro': raw_data_anthro,
    'dietary': raw_data_dietary,
    'biochemical': raw_data_biochemical,
    'socio': raw_data_socio
}.items():
    print(f'{name:10s} shape={df.shape}')

save_ckpt('03_raw_data', {
    'clinical': raw_data_clinical,
    'anthro': raw_data_anthro,
    'dietary': raw_data_dietary,
    'biochemical': raw_data_biochemical,
    'socio': raw_data_socio,
})
mark_stage('03_read_datasets_done')

clinical   shape=(351701, 29)
anthro     shape=(444400, 18)
dietary    shape=(234293, 75)
biochemical shape=(161643, 13)
socio      shape=(654425, 20)


## 4) Merge datasets

In [5]:
MERGE_KEYS = ['enns_year', 'hhnum', 'member_code']

def merge_no_overlap(base_df, right_df, keys=MERGE_KEYS):
    overlap = (set(base_df.columns) & set(right_df.columns)) - set(keys)
    right_clean = right_df.drop(columns=list(overlap), errors='ignore')
    return base_df.merge(right_clean, on=keys, how='left')

merged_data = raw_data_clinical.copy()
merged_data = merge_no_overlap(merged_data, raw_data_anthro)
merged_data = merge_no_overlap(merged_data, raw_data_dietary)
merged_data = merge_no_overlap(merged_data, raw_data_biochemical)
merged_data = merge_no_overlap(merged_data, raw_data_socio)

print('Merged shape:', merged_data.shape)
save_ckpt('04_merged_data', merged_data)
mark_stage('04_merge_done')

Merged shape: (351701, 115)


## 5) Target creation + core feature engineering

In [6]:
bp_cols = [c for c in merged_data.columns if 'SBP' in c.upper() or 'DBP' in c.upper()]
sbp_col = next(c for c in bp_cols if 'SBP' in c.upper())
dbp_col = next(c for c in bp_cols if 'DBP' in c.upper())

merged_data['Hypertension'] = ((merged_data[sbp_col] >= 130) | (merged_data[dbp_col] >= 80)).astype(int)

if {'weight','height'}.issubset(set(merged_data.columns)):
    merged_data['BMI'] = merged_data['weight'] / ((merged_data['height'] / 100) ** 2)

print('Target prevalence (%):', round(100 * merged_data['Hypertension'].mean(), 2))

Target prevalence (%): 32.68


## 6) Data quality checks (often skipped)

In [7]:
print('Rows before dedup:', len(merged_data))
merged_data = merged_data.drop_duplicates(subset=MERGE_KEYS)
print('Rows after dedup :', len(merged_data))

missing_pct = merged_data.isna().mean().sort_values(ascending=False) * 100
display(missing_pct.head(20).to_frame('missing_%'))

Rows before dedup: 351701
Rows after dedup : 351701


,missing_%
mos_preg,98.969295
mos_lactation,96.900492
chol,87.594860
ldl,87.594576
tri,87.594292
hdl,87.594007
vita,84.054069
uic,78.796762
fbs,78.126590
hemoglobin,64.948920


## 7) Domain filters + dropping admin/leakage columns

In [8]:
# Adults only if available
if 'anthro_group' in merged_data.columns:
    merged_data = merged_data[merged_data['anthro_group'].isin([1, 4])].copy()

drop_cols = [
    sbp_col, dbp_col,
    'enns_year', 'hhnum', 'member_code',
    'intdate', 'enumcode', 'interview_status',
    'regcode', 'provhuc', 'ms_psucode',
    'fwgth_natl_var', 'fwgth_prov', 'fwgth_natl2_var',
    'fwgti_natl_var', 'fwgti_prov', 'fwgti_natl2_var', 'fwgti_prov2',
    'rep_natl', 'rep_prov'
]
merged_data = merged_data.drop(columns=[c for c in drop_cols if c in merged_data.columns], errors='ignore')

print('Shape after filters/drops:', merged_data.shape)

Shape after filters/drops: (236247, 102)


## 8) Categorical handling + train/cal/test split

In [29]:
TARGET_COL = 'Hypertension'
y_full = merged_data[TARGET_COL].astype(int)
X_full = merged_data.drop(columns=[TARGET_COL])

def _add_domain_composites(df):
    out = df.copy()

    def _numeric_cols_by_keywords(keywords):
        cols = []
        for c in out.columns:
            cl = c.lower()
            if any(k in cl for k in keywords) and pd.api.types.is_numeric_dtype(out[c]):
                cols.append(c)
        return cols

    def _add_zscore_sum(cols, new_col):
        if not cols:
            return
        z = (out[cols] - out[cols].mean()) / out[cols].std(ddof=0).replace(0, 1)
        out[new_col] = z.fillna(0.0).sum(axis=1)

    metabolic_cols = _numeric_cols_by_keywords(['bmi', 'waist', 'chol', 'ldl', 'hdl', 'tri', 'fbs', 'glucose'])
    kidney_cols = _numeric_cols_by_keywords(['creat', 'egfr', 'bun', 'uric'])
    lifestyle_cols = _numeric_cols_by_keywords(['alcohol', 'smok', 'cig', 'salt', 'sodium'])

    _add_zscore_sum(metabolic_cols, 'risk_metabolic_proxy')
    _add_zscore_sum(kidney_cols, 'risk_kidney_proxy')
    _add_zscore_sum(lifestyle_cols, 'risk_lifestyle_proxy')

    if 'risk_metabolic_proxy' in out.columns and 'risk_lifestyle_proxy' in out.columns:
        out['risk_meta_x_lifestyle'] = out['risk_metabolic_proxy'] * out['risk_lifestyle_proxy']

    return out

n0 = X_full.shape[1]
X_full = _add_domain_composites(X_full)
print(f'Domain composite features added: {X_full.shape[1] - n0}')

obj_cols = X_full.select_dtypes(include=['object', 'category']).columns.tolist()
X_full = pd.get_dummies(X_full, columns=obj_cols, drop_first=False, dummy_na=True)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X_full, y_full, test_size=0.30, random_state=RANDOM_SEED, stratify=y_full
)
X_cal, X_te, y_cal, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_tmp
)

print('Train:', X_tr.shape, '| Cal:', X_cal.shape, '| Test:', X_te.shape)

Domain composite features added: 3
Train: (165372, 104) | Cal: (35437, 104) | Test: (35438, 104)


## 9) Imputation + scaling

In [30]:
def _enrich_features(Xtr_in, Xcal_in, Xte_in):
    Xtr_e = Xtr_in.copy()
    Xcal_e = Xcal_in.copy()
    Xte_e = Xte_in.copy()

    # Row-level missingness features
    for df in (Xtr_e, Xcal_e, Xte_e):
        df['row_missing_count'] = df.isna().sum(axis=1).astype(float)
        df['row_missing_ratio'] = df.isna().mean(axis=1).astype(float)

    # Missing indicators for high-missing columns (bounded to avoid feature explosion)
    miss_rate = Xtr_e.isna().mean().sort_values(ascending=False)
    miss_cols = miss_rate[miss_rate > 0.20].index.tolist()[:20]
    for c in miss_cols:
        ind_col = f'miss_ind__{c}'
        Xtr_e[ind_col] = Xtr_e[c].isna().astype(np.int8)
        Xcal_e[ind_col] = Xcal_e[c].isna().astype(np.int8)
        Xte_e[ind_col] = Xte_e[c].isna().astype(np.int8)

    # Light interaction block from continuous-like columns
    cont_cols = [c for c in Xtr_e.columns if Xtr_e[c].nunique(dropna=True) > 20]
    cont_cols = cont_cols[:8]
    for i in range(min(len(cont_cols), 4)):
        for j in range(i + 1, min(len(cont_cols), 4)):
            c1, c2 = cont_cols[i], cont_cols[j]
            prod_col = f'int_prod__{c1}__{c2}'
            ratio_col = f'int_ratio__{c1}__{c2}'
            for df in (Xtr_e, Xcal_e, Xte_e):
                df[prod_col] = df[c1] * df[c2]
                df[ratio_col] = df[c1] / (df[c2].abs() + 1e-6)

    return Xtr_e, Xcal_e, Xte_e

X_tr, X_cal, X_te = _enrich_features(X_tr, X_cal, X_te)
print(f'After enrichment: Xtr={X_tr.shape}, Xcal={X_cal.shape}, Xte={X_te.shape}')

num_cols = X_tr.columns.tolist()
print(f'Input matrices shapes: Xtr={X_tr.shape}, Xcal={X_cal.shape}, Xte={X_te.shape}')
print(f'Input num_cols: {len(num_cols)} columns')

# Pre-fill problematic columns
problem_cols = []
for col in X_tr.columns:
    if X_tr[col].isna().all():
        problem_cols.append(f'{col} (all NaN)')
        X_tr[col] = 0.0
        X_cal[col] = 0.0
        X_te[col] = 0.0
    elif X_tr[col].nunique() <= 1 and X_tr[col].isna().sum() > 0:
        problem_cols.append(f'{col} (constant + NaN)')
        fill_val = X_tr[col].dropna().iloc[0] if len(X_tr[col].dropna()) > 0 else 0.0
        X_tr[col] = X_tr[col].fillna(fill_val)
        X_cal[col] = X_cal[col].fillna(fill_val)
        X_te[col] = X_te[col].fillna(fill_val)

if problem_cols:
    print(f'Found {len(problem_cols)} problematic columns: {problem_cols[:5]}...')
else:
    print('No problematic columns found')

# Use KNNImputer with error handling
imputer = KNNImputer(n_neighbors=11)
Xtr_imp_arr = imputer.fit_transform(X_tr)

# Handle mismatch in column counts (edge case with certain sklearn/data combinations)
if Xtr_imp_arr.shape[1] != len(num_cols):
    print(f'WARNING: Imputer returned {Xtr_imp_arr.shape[1]} columns but expected {len(num_cols)}')
    print('Taking first N columns from input to match imputer output')
    num_cols = num_cols[:Xtr_imp_arr.shape[1]]

print(f'Imputer output shape: {Xtr_imp_arr.shape}, columns being used: {len(num_cols)}')

Xtr_imp = pd.DataFrame(Xtr_imp_arr, columns=num_cols, index=X_tr.index)
Xcal_imp = pd.DataFrame(imputer.transform(X_cal), columns=num_cols, index=X_cal.index)
Xte_imp = pd.DataFrame(imputer.transform(X_te), columns=num_cols, index=X_te.index)

scaler = StandardScaler()
Xtr = pd.DataFrame(scaler.fit_transform(Xtr_imp), columns=num_cols, index=Xtr_imp.index)
Xcal = pd.DataFrame(scaler.transform(Xcal_imp), columns=num_cols, index=Xcal_imp.index)
Xte = pd.DataFrame(scaler.transform(Xte_imp), columns=num_cols, index=Xte_imp.index)

# Mutual Information-Based mRMR feature selection
USE_MRMR = True
MRMR_K = 90
MRMR_PREFILTER = 240
MRMR_REDUNDANCY_WEIGHT = 0.20

def _mrmr_select(X, y, k=80, prefilter=200, redundancy_weight=0.2):
    n_features = X.shape[1]
    k = min(k, n_features)
    if n_features == 0:
        return [], pd.Series(dtype=float)

    mi = mutual_info_classif(X, y, random_state=RANDOM_SEED)
    mi_scores = pd.Series(mi, index=X.columns).fillna(0.0)
    pool = mi_scores.sort_values(ascending=False).head(min(prefilter, n_features)).index.tolist()
    if not pool:
        return X.columns.tolist(), mi_scores

    selected = [pool[0]]
    remaining = set(pool[1:])
    corr = X[pool].corr().abs().fillna(0.0)

    target_size = min(k, len(pool))
    while len(selected) < target_size and remaining:
        best_feat = None
        best_score = -1e18
        for feat in list(remaining):
            redundancy = float(corr.loc[feat, selected].mean()) if selected else 0.0
            score = float(mi_scores[feat] - redundancy_weight * redundancy)
            if score > best_score:
                best_score = score
                best_feat = feat
        selected.append(best_feat)
        remaining.remove(best_feat)

    return selected, mi_scores

if USE_MRMR:
    selected_cols, mi_scores = _mrmr_select(
        Xtr, y_tr, k=MRMR_K, prefilter=MRMR_PREFILTER, redundancy_weight=MRMR_REDUNDANCY_WEIGHT
    )
    Xtr = Xtr[selected_cols].copy()
    Xcal = Xcal[selected_cols].copy()
    Xte = Xte[selected_cols].copy()
    print(f'mRMR enabled: selected {len(selected_cols)} features from {len(num_cols)}')
    print('Top selected features (first 20):', selected_cols[:20])
else:
    selected_cols = Xtr.columns.tolist()

print('Prepared matrices:', Xtr.shape, Xcal.shape, Xte.shape)
save_ckpt('09_prepared_matrices', {
    'Xtr': Xtr, 'Xcal': Xcal, 'Xte': Xte,
    'y_tr': y_tr, 'y_cal': y_cal, 'y_te': y_te,
    'selected_cols': selected_cols
})
mark_stage('09_preprocessing_done')


After enrichment: Xtr=(165372, 138), Xcal=(35437, 138), Xte=(35438, 138)
Input matrices shapes: Xtr=(165372, 138), Xcal=(35437, 138), Xte=(35438, 138)
Input num_cols: 138 columns
Found 2 problematic columns: ['mos_lactation (all NaN)', 'mos_preg (all NaN)']...
Imputer output shape: (165372, 138), columns being used: 138
mRMR enabled: selected 90 features from 138
Top selected features (first 20): ['int_prod__fbs__tri', 'int_prod__drnk_30d_num__fbs', 'fg13', 'epwt_fg7', 'ldl', 'rhc', 'epwt_fg18', 'waist', 'epwt_fg17', 'int_prod__drnk_30d_num__tri', 'int_prod__chol__tri', 'int_ratio__fbs__chol', 'uic', 'fg19', 'epwt_fg5', 'age', 'hdl', 'int_prod__fbs__chol', 'fg26', 'int_ratio__drnk_30d_num__chol']
Prepared matrices: (165372, 90) (35437, 90) (35438, 90)


## 10) Model training (XGB, LGBM, CAT)

In [47]:
target_accuracy = 0.80
target_recall = 0.80

# Hyperparameter search controls
RUN_GRID_SEARCH = True
RUN_RANDOM_SEARCH = True
RUN_BAYESIAN_OPT = True
RUN_GENETIC_SEARCH = True

RANDOM_SEARCH_ITERS = 80
BAYESIAN_TRIAL_SCHEDULE = [24, 48, 96]
GENETIC_POP_SIZE = 20
GENETIC_GENERATIONS = 8
GENETIC_MUTATION_RATE = 0.20

pos_rate = float(y_tr.mean())
pos_majority = pos_rate >= 0.60
spw = (y_tr.eq(0).sum() / max(y_tr.eq(1).sum(), 1))
if pos_majority:
    spw = 1.0

xgb_gpu_params = {'device': 'cuda', 'tree_method': 'hist'} if XGB_GPU_AVAILABLE else {'tree_method': 'hist'}
lgbm_gpu_params = {'device_type': 'gpu'} if LGBM_GPU_AVAILABLE else {}
cat_gpu_params = {'task_type': 'GPU'} if CAT_GPU_AVAILABLE else {}

lgbm_class_weight = None if pos_majority else 'balanced'
cat_weight_params = {} if pos_majority else {'auto_class_weights': 'Balanced'}

base_models = {
    'XGB_bal': XGBClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, eval_metric='logloss',
        scale_pos_weight=spw, random_state=RANDOM_SEED, verbosity=0,
        **xgb_gpu_params
    ),
    'LGB_bal': LGBMClassifier(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8, class_weight=lgbm_class_weight,
        random_state=RANDOM_SEED, verbose=-1,
        **lgbm_gpu_params
    ),
    'CAT_bal': CatBoostClassifier(
        iterations=500, learning_rate=0.05, depth=6,
        random_seed=RANDOM_SEED, verbose=0,
        **cat_weight_params,
        **cat_gpu_params
    ),
}

def _choose_threshold_for_dual_target(y_true, p, min_acc=0.80, min_rec=0.80):
    rows = []
    for thr in np.linspace(0.10, 0.90, 81):
        y_hat = (p >= thr).astype(int)
        acc = accuracy_score(y_true, y_hat)
        rec = recall_score(y_true, y_hat, zero_division=0)
        f1 = f1_score(y_true, y_hat, zero_division=0)
        rows.append({'threshold': thr, 'accuracy': acc, 'recall': rec, 'f1': f1, 'dual_min': min(acc, rec)})
    sweep = pd.DataFrame(rows)
    feasible = sweep[(sweep['accuracy'] >= min_acc) & (sweep['recall'] >= min_rec)]
    if not feasible.empty:
        best = feasible.sort_values(['recall', 'accuracy', 'f1'], ascending=False).iloc[0]
        mode = 'dual_target_met'
    else:
        best = sweep.sort_values(['dual_min', 'recall', 'accuracy', 'f1'], ascending=False).iloc[0]
        mode = 'fallback_best_dual_min'
    return float(best['threshold']), best, mode

# Unified evaluator for all search strategies
def _evaluate_lgbm_params(params):
    cfg = {
        **params,
        'random_state': RANDOM_SEED,
        'class_weight': lgbm_class_weight,
        'verbose': -1,
        **lgbm_gpu_params,
    }
    mdl = LGBMClassifier(**cfg)
    mdl.fit(Xtr, y_tr)
    p_cal = mdl.predict_proba(Xcal)[:, 1]
    thr, best_row, mode = _choose_threshold_for_dual_target(
        y_cal, p_cal, min_acc=target_accuracy, min_rec=target_recall
    )
    bonus = 1.0 if mode == 'dual_target_met' else 0.0
    score = float(best_row['dual_min']) + bonus
    return {
        'params': params,
        'score': score,
        'threshold': float(thr),
        'accuracy': float(best_row['accuracy']),
        'recall': float(best_row['recall']),
        'dual_min': float(best_row['dual_min']),
        'mode': mode,
    }

# Parameter spaces
grid_space = {
    'n_estimators': [300, 500, 800],
    'learning_rate': [0.02, 0.05, 0.08],
    'num_leaves': [31, 63, 127],
    'max_depth': [5, 8],
    'min_child_samples': [20, 60],
    'subsample': [0.7, 0.9],
    'colsample_bytree': [0.7, 0.9],
    'reg_alpha': [1e-4, 1e-2],
    'reg_lambda': [1e-4, 1e-2],
}

random_space = {
    'n_estimators': list(range(250, 1201, 50)),
    'learning_rate': [0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12],
    'num_leaves': [31, 47, 63, 95, 127, 191, 255, 320],
    'max_depth': [3, 4, 5, 6, 8, 10, 12, 14],
    'min_child_samples': [8, 12, 20, 40, 60, 80, 120, 160],
    'subsample': [0.55, 0.65, 0.75, 0.85, 0.95, 1.0],
    'colsample_bytree': [0.55, 0.65, 0.75, 0.85, 0.95, 1.0],
    'reg_alpha': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 20.0],
    'reg_lambda': [1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 20.0],
}

search_results = []
method_best = {}

# 1) Grid Search (exhaustive)
if RUN_GRID_SEARCH:
    print('Running Grid Search (exhaustive)...')
    best_grid = None
    grid_count = 0
    for params in ParameterGrid(grid_space):
        grid_count += 1
        res = _evaluate_lgbm_params(params)
        res['method'] = 'grid_search'
        search_results.append(res)
        if (best_grid is None) or (res['score'] > best_grid['score']):
            best_grid = res
    method_best['grid_search'] = best_grid
    print(f'Grid Search done: {grid_count} combinations | best_score={best_grid["score"]:.4f} | mode={best_grid["mode"]}')

# 2) Randomized Search
if RUN_RANDOM_SEARCH:
    print('Running Randomized Search...')
    best_random = None
    sampler = ParameterSampler(random_space, n_iter=RANDOM_SEARCH_ITERS, random_state=RANDOM_SEED)
    for params in sampler:
        res = _evaluate_lgbm_params(params)
        res['method'] = 'random_search'
        search_results.append(res)
        if (best_random is None) or (res['score'] > best_random['score']):
            best_random = res
    method_best['random_search'] = best_random
    print(f'Randomized Search done: {RANDOM_SEARCH_ITERS} samples | best_score={best_random["score"]:.4f} | mode={best_random["mode"]}')

# 3) Bayesian Optimization (Optuna)
if RUN_BAYESIAN_OPT:
    print('Running Bayesian Optimization...')
    best_bayes = None

    def _optuna_objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 250, 1200),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.15, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 320),
            'max_depth': trial.suggest_int('max_depth', 3, 14),
            'min_child_samples': trial.suggest_int('min_child_samples', 8, 160),
            'subsample': trial.suggest_float('subsample', 0.55, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.55, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 20.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 20.0, log=True),
        }
        res = _evaluate_lgbm_params(params)
        trial.set_user_attr('result', res)
        return res['score']

    for round_idx, n_trials in enumerate(BAYESIAN_TRIAL_SCHEDULE, start=1):
        print(f'  Bayesian round {round_idx}/{len(BAYESIAN_TRIAL_SCHEDULE)}: {n_trials} trials')
        study = optuna.create_study(
            direction='maximize',
            sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED + round_idx),
        )
        study.optimize(_optuna_objective, n_trials=int(n_trials), show_progress_bar=False)

        round_best = dict(study.best_trial.user_attrs['result'])
        round_best['method'] = 'bayesian_optimization'
        search_results.append(round_best)
        if (best_bayes is None) or (round_best['score'] > best_bayes['score']):
            best_bayes = round_best

        print(f"    round best_score={round_best['score']:.4f} | mode={round_best['mode']}")
        if round_best['mode'] == 'dual_target_met':
            print('    Dual target reached in Bayesian optimization; stopping early.')
            break

    method_best['bayesian_optimization'] = best_bayes
    print(f"Bayesian Optimization best_score={best_bayes['score']:.4f} | mode={best_bayes['mode']}")

# 4) Genetic Algorithm
if RUN_GENETIC_SEARCH:
    print('Running Genetic Algorithm search...')
    rng = np.random.default_rng(RANDOM_SEED)
    genes = list(random_space.keys())
    choices = {k: list(v) for k, v in random_space.items()}
    fitness_cache = {}

    def _make_individual():
        return {k: choices[k][rng.integers(0, len(choices[k]))] for k in genes}

    def _key(ind):
        return tuple((k, ind[k]) for k in genes)

    def _fitness(ind):
        k = _key(ind)
        if k not in fitness_cache:
            fitness_cache[k] = _evaluate_lgbm_params(ind)
        return fitness_cache[k]

    def _mutate(ind, rate=0.20):
        out = dict(ind)
        for g in genes:
            if rng.random() < rate:
                out[g] = choices[g][rng.integers(0, len(choices[g]))]
        return out

    def _crossover(a, b):
        child = {}
        for g in genes:
            child[g] = a[g] if rng.random() < 0.5 else b[g]
        return child

    population = [_make_individual() for _ in range(GENETIC_POP_SIZE)]
    best_genetic = None

    for gen in range(1, GENETIC_GENERATIONS + 1):
        scored = []
        for ind in population:
            res = _fitness(ind)
            scored.append((res['score'], ind, res))
        scored.sort(key=lambda x: x[0], reverse=True)

        gen_best = dict(scored[0][2])
        gen_best['method'] = 'genetic_algorithm'
        search_results.append(gen_best)

        if (best_genetic is None) or (gen_best['score'] > best_genetic['score']):
            best_genetic = gen_best

        print(f"  generation {gen}/{GENETIC_GENERATIONS} | best_score={gen_best['score']:.4f} | mode={gen_best['mode']}")
        if gen_best['mode'] == 'dual_target_met':
            print('  Dual target reached in genetic search; stopping early.')
            break

        elite_n = max(2, int(0.25 * GENETIC_POP_SIZE))
        elites = [dict(scored[i][1]) for i in range(elite_n)]
        new_population = elites.copy()

        while len(new_population) < GENETIC_POP_SIZE:
            i1 = rng.integers(0, elite_n)
            i2 = rng.integers(0, elite_n)
            parent_a, parent_b = elites[i1], elites[i2]
            child = _crossover(parent_a, parent_b)
            child = _mutate(child, rate=GENETIC_MUTATION_RATE)
            new_population.append(child)

        population = new_population

    method_best['genetic_algorithm'] = best_genetic
    print(f"Genetic Algorithm best_score={best_genetic['score']:.4f} | mode={best_genetic['mode']}")

method_best_rows = []
for method_name, best_res in method_best.items():
    if best_res is None:
        continue
    method_best_rows.append({
        'method': method_name,
        'score': best_res['score'],
        'dual_min': best_res['dual_min'],
        'accuracy': best_res['accuracy'],
        'recall': best_res['recall'],
        'threshold': best_res['threshold'],
        'mode': best_res['mode'],
    })

if not method_best_rows:
    raise RuntimeError('No hyperparameter search method produced results.')

method_comparison = pd.DataFrame(method_best_rows).sort_values(
    ['score', 'dual_min', 'recall', 'accuracy'], ascending=False
).reset_index(drop=True)
display(method_comparison)

best_method_name = method_comparison.iloc[0]['method']
best_method_result = method_best[best_method_name]
best_lgb_params = dict(best_method_result['params'])
best_lgb_params.update({
    'random_state': RANDOM_SEED,
    'class_weight': lgbm_class_weight,
    'verbose': -1,
    **lgbm_gpu_params,
})

base_models['LGB_tuned_best'] = LGBMClassifier(**best_lgb_params)
print(f'Best search method: {best_method_name} | score={best_method_result["score"]:.4f} | mode={best_method_result["mode"]}')

# Keep stacker GPU-safe: avoid CatBoost inside internal CV and run serially.
stack_model = StackingClassifier(
    estimators=[
        ('xgb', base_models['XGB_bal']),
        ('lgb', base_models['LGB_bal']),
        ('lgb_tuned', base_models['LGB_tuned_best']),
    ],
    final_estimator=LogisticRegression(max_iter=2000),
    stack_method='predict_proba',
    cv=3,
    n_jobs=1,
)

candidate_models = {**base_models, 'STACK_blend': stack_model}

def _run_nested_cv_feasibility(models, X, y, min_acc=0.80, min_rec=0.80, outer_splits=3):
    rows = []
    outer_cv = StratifiedKFold(n_splits=outer_splits, shuffle=True, random_state=RANDOM_SEED)

    for model_name, model in models.items():
        fold_metrics = []
        for fold, (outer_tr_idx, outer_te_idx) in enumerate(outer_cv.split(X, y), start=1):
            X_outer_tr = X.iloc[outer_tr_idx]
            y_outer_tr = y.iloc[outer_tr_idx]
            X_outer_te = X.iloc[outer_te_idx]
            y_outer_te = y.iloc[outer_te_idx]

            X_inner_tr, X_inner_va, y_inner_tr, y_inner_va = train_test_split(
                X_outer_tr, y_outer_tr, test_size=0.25, random_state=RANDOM_SEED + fold, stratify=y_outer_tr
            )

            mdl_inner = clone(model)
            mdl_inner.fit(X_inner_tr, y_inner_tr)
            p_inner_va = mdl_inner.predict_proba(X_inner_va)[:, 1]
            thr, _, _ = _choose_threshold_for_dual_target(
                y_inner_va, p_inner_va, min_acc=min_acc, min_rec=min_rec
            )

            mdl_outer = clone(model)
            mdl_outer.fit(X_outer_tr, y_outer_tr)
            p_outer_te = mdl_outer.predict_proba(X_outer_te)[:, 1]
            y_hat = (p_outer_te >= thr).astype(int)
            acc = accuracy_score(y_outer_te, y_hat)
            rec = recall_score(y_outer_te, y_hat, zero_division=0)
            fold_metrics.append({
                'fold': fold,
                'accuracy': acc,
                'recall': rec,
                'dual_min': min(acc, rec),
                'meets_dual_target': int((acc >= min_acc) and (rec >= min_rec)),
            })

        fdf = pd.DataFrame(fold_metrics)
        rows.append({
            'model': model_name,
            'nested_cv_accuracy_mean': float(fdf['accuracy'].mean()),
            'nested_cv_recall_mean': float(fdf['recall'].mean()),
            'nested_cv_dual_min_mean': float(fdf['dual_min'].mean()),
            'nested_cv_dual_target_hit_rate': float(fdf['meets_dual_target'].mean()),
        })

    return pd.DataFrame(rows).sort_values(
        ['nested_cv_dual_target_hit_rate', 'nested_cv_dual_min_mean', 'nested_cv_recall_mean', 'nested_cv_accuracy_mean'],
        ascending=False
    ).reset_index(drop=True)

print('Running nested CV feasibility report...')
nested_cv_report = _run_nested_cv_feasibility(
    candidate_models, Xtr, y_tr, min_acc=target_accuracy, min_rec=target_recall, outer_splits=3
)
display(nested_cv_report)

class OOFCalibratedStack:
    def __init__(self, base_model_factories, random_seed=42):
        self.base_model_factories = base_model_factories
        self.random_seed = random_seed
        self.base_models_full_ = {}
        self.meta_ = LogisticRegression(max_iter=2000)
        self.calibrator_ = LogisticRegression(max_iter=2000)

    def fit(self, X_train, y_train, X_calib, y_calib):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=self.random_seed)
        model_names = list(self.base_model_factories.keys())
        oof = np.zeros((len(X_train), len(model_names)), dtype=float)

        for m_idx, name in enumerate(model_names):
            for tr_idx, va_idx in cv.split(X_train, y_train):
                mdl = clone(self.base_model_factories[name])
                mdl.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
                oof[va_idx, m_idx] = mdl.predict_proba(X_train.iloc[va_idx])[:, 1]

            full_mdl = clone(self.base_model_factories[name])
            full_mdl.fit(X_train, y_train)
            self.base_models_full_[name] = full_mdl

        self.meta_.fit(oof, y_train)

        cal_stack = np.column_stack([
            self.base_models_full_[name].predict_proba(X_calib)[:, 1]
            for name in model_names
        ])
        p_meta_cal = self.meta_.predict_proba(cal_stack)[:, 1]
        self.calibrator_.fit(p_meta_cal.reshape(-1, 1), y_calib)
        return self

    def predict_proba(self, X):
        model_names = list(self.base_models_full_.keys())
        stack = np.column_stack([
            self.base_models_full_[name].predict_proba(X)[:, 1]
            for name in model_names
        ])
        p_meta = self.meta_.predict_proba(stack)[:, 1]
        p_cal = self.calibrator_.predict_proba(p_meta.reshape(-1, 1))[:, 1]
        return np.column_stack([1 - p_cal, p_cal])

class TwoStageRecallGate:
    def __init__(self, stage1_model, stage2_model=None, stage1_recall_target=0.90):
        self.stage1_model = stage1_model
        self.stage2_model = stage2_model if stage2_model is not None else LogisticRegression(max_iter=2000)
        self.stage1_recall_target = stage1_recall_target
        self.stage1_threshold_ = 0.5
        self._stage2_fitted = False

    def fit(self, X_train, y_train, X_calib, y_calib):
        self.stage1_model.fit(X_train, y_train)
        p_cal = self.stage1_model.predict_proba(X_calib)[:, 1]
        grid_rows = []
        for t in np.linspace(0.10, 0.90, 81):
            y_hat = (p_cal >= t).astype(int)
            rec = recall_score(y_calib, y_hat, zero_division=0)
            pre = precision_score(y_calib, y_hat, zero_division=0)
            grid_rows.append((t, rec, pre))
        grid_df = pd.DataFrame(grid_rows, columns=['thr', 'recall', 'precision'])
        feasible = grid_df[grid_df['recall'] >= self.stage1_recall_target]
        if not feasible.empty:
            best = feasible.sort_values(['precision', 'recall'], ascending=False).iloc[0]
        else:
            best = grid_df.sort_values(['recall', 'precision'], ascending=False).iloc[0]
        self.stage1_threshold_ = float(best['thr'])

        p_train = self.stage1_model.predict_proba(X_train)[:, 1]
        gate_mask = p_train >= self.stage1_threshold_
        if int(gate_mask.sum()) >= 500 and y_train[gate_mask].nunique() > 1:
            X_stage2 = X_train.loc[gate_mask].copy()
            X_stage2['p_stage1'] = p_train[gate_mask]
            self.stage2_model.fit(X_stage2, y_train[gate_mask])
            self._stage2_fitted = True
        else:
            self._stage2_fitted = False
        return self

    def predict_proba(self, X):
        p1 = self.stage1_model.predict_proba(X)[:, 1]
        p_final = p1.copy()
        gate_mask = p1 >= self.stage1_threshold_
        if self._stage2_fitted and int(gate_mask.sum()) > 0:
            X_stage2 = X.loc[gate_mask].copy()
            X_stage2['p_stage1'] = p1[gate_mask]
            p2 = self.stage2_model.predict_proba(X_stage2)[:, 1]
            p_final[gate_mask] = 0.35 * p1[gate_mask] + 0.65 * p2
        p_final = np.clip(p_final, 1e-6, 1 - 1e-6)
        return np.column_stack([1 - p_final, p_final])

cv_rows = []
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

for model_name, model in candidate_models.items():
    fold_stats = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(Xtr, y_tr), start=1):
        X_tr_f = Xtr.iloc[tr_idx]
        y_tr_f = y_tr.iloc[tr_idx]
        X_va_f = Xtr.iloc[va_idx]
        y_va_f = y_tr.iloc[va_idx]

        mdl = clone(model)
        mdl.fit(X_tr_f, y_tr_f)
        p_va = mdl.predict_proba(X_va_f)[:, 1]

        thr, best_row, mode = _choose_threshold_for_dual_target(
            y_va_f, p_va, min_acc=target_accuracy, min_rec=target_recall
        )
        fold_stats.append({
            'fold': fold,
            'threshold': thr,
            'accuracy': float(best_row['accuracy']),
            'recall': float(best_row['recall']),
            'f1': float(best_row['f1']),
            'dual_min': float(best_row['dual_min']),
            'meets_dual_target': int(mode == 'dual_target_met')
        })

    fold_df = pd.DataFrame(fold_stats)
    cv_rows.append({
        'model': model_name,
        'cv_accuracy_mean': float(fold_df['accuracy'].mean()),
        'cv_recall_mean': float(fold_df['recall'].mean()),
        'cv_f1_mean': float(fold_df['f1'].mean()),
        'cv_dual_min_mean': float(fold_df['dual_min'].mean()),
        'cv_dual_target_hit_rate': float(fold_df['meets_dual_target'].mean()),
        'cv_threshold_mean': float(fold_df['threshold'].mean())
    })

cv_feasibility = pd.DataFrame(cv_rows).sort_values(
    ['cv_dual_target_hit_rate', 'cv_dual_min_mean', 'cv_recall_mean', 'cv_accuracy_mean'],
    ascending=False
).reset_index(drop=True)

display(cv_feasibility)
if (cv_feasibility['cv_dual_target_hit_rate'] > 0).any():
    print('At least one model reached 0.80/0.80 on some CV folds.')
else:
    print('No model reached 0.80/0.80 in CV folds; this target may be infeasible without stronger signal/features.')

fitted_models = {}
for name, mdl in candidate_models.items():
    mdl.fit(Xtr, y_tr)
    fitted_models[name] = mdl

oof_base_factories = {
    'xgb': fitted_models['XGB_bal'],
    'lgb': fitted_models['LGB_bal'],
    'lgb_tuned': fitted_models['LGB_tuned_best'],
}
oof_stack = OOFCalibratedStack(oof_base_factories, random_seed=RANDOM_SEED)
oof_stack.fit(Xtr, y_tr, Xcal, y_cal)
fitted_models['OOF_meta_cal'] = oof_stack

two_stage = TwoStageRecallGate(
    stage1_model=clone(fitted_models['LGB_tuned_best']),
    stage2_model=LogisticRegression(max_iter=2000),
    stage1_recall_target=0.90,
)
two_stage.fit(Xtr, y_tr, Xcal, y_cal)
fitted_models['TwoStage_gate'] = two_stage

print('Trained models:', list(fitted_models.keys()))
print(f'Train positive rate: {pos_rate:.3f} | Positive-majority mode: {pos_majority}')
print(f'Targets -> accuracy>={target_accuracy:.2f}, recall>={target_recall:.2f}')
print(f"GPU used -> XGB:{XGB_GPU_AVAILABLE} | LGBM:{LGBM_GPU_AVAILABLE} | CAT:{CAT_GPU_AVAILABLE}")

Running Grid Search (exhaustive)...
Grid Search done: 1728 combinations | best_score=0.6676 | mode=fallback_best_dual_min
Running Randomized Search...


[I 2026-03-26 18:54:50,443] A new study created in memory with name: no-name-e2e3cd18-80a5-4cce-8574-b04446e9294e


Randomized Search done: 80 samples | best_score=0.6669 | mode=fallback_best_dual_min
Running Bayesian Optimization...
  Bayesian round 1/3: 24 trials


[I 2026-03-26 18:54:55,011] Trial 0 finished with value: 0.6666246930680602 and parameters: {'n_estimators': 359, 'learning_rate': 0.039685799603129755, 'num_leaves': 69, 'max_depth': 5, 'min_child_samples': 58, 'subsample': 0.9366118709268689, 'colsample_bytree': 0.8497405958941116, 'reg_alpha': 0.025696817024761818, 'reg_lambda': 1.5234106170718591e-05}. Best is trial 0 with value: 0.6666246930680602.
[I 2026-03-26 18:55:13,579] Trial 1 finished with value: 0.6648700510765584 and parameters: {'n_estimators': 947, 'learning_rate': 0.01915839270563527, 'num_leaves': 263, 'max_depth': 6, 'min_child_samples': 16, 'subsample': 0.9399918884046401, 'colsample_bytree': 0.6494630481479671, 'reg_alpha': 0.0035632858239925103, 'reg_lambda': 0.0009811453398205854}. Best is trial 0 with value: 0.6666246930680602.
[I 2026-03-26 18:55:39,456] Trial 2 finished with value: 0.6553547818422213 and parameters: {'n_estimators': 322, 'learning_rate': 0.08800657561647202, 'num_leaves': 277, 'max_depth': 14

    round best_score=0.6674 | mode=fallback_best_dual_min
  Bayesian round 2/3: 48 trials


...

,method,score,dual_min,accuracy,recall,threshold,mode
0,genetic_algorithm,0.667918,0.667918,0.667918,0.670528,0.53,fallback_best_dual_min
1,bayesian_optimization,0.667889,0.667889,0.667889,0.668451,0.53,fallback_best_dual_min
2,grid_search,0.667579,0.667579,0.667579,0.682931,0.52,fallback_best_dual_min
3,random_search,0.666874,0.666874,0.666874,0.668765,0.53,fallback_best_dual_min


Best search method: genetic_algorithm | score=0.6679 | mode=fallback_best_dual_min
Running nested CV feasibility report...


,model,nested_cv_accuracy_mean,nested_cv_recall_mean,nested_cv_dual_min_mean,nested_cv_dual_target_hit_rate
0,CAT_bal,0.662730,0.679357,0.662730,0.0
1,LGB_tuned_best,0.662537,0.679708,0.662537,0.0
2,STACK_blend,0.662651,0.679992,0.661829,0.0
3,XGB_bal,0.661732,0.671802,0.661732,0.0
4,LGB_bal,0.660372,0.675027,0.660372,0.0


,model,cv_accuracy_mean,cv_recall_mean,cv_f1_mean,cv_dual_min_mean,cv_dual_target_hit_rate,cv_threshold_mean
0,STACK_blend,0.664048,0.678831,0.644250,0.664048,0.0,0.472
1,CAT_bal,0.663861,0.674716,0.642738,0.663861,0.0,0.526
2,LGB_tuned_best,0.663782,0.677536,0.643654,0.663782,0.0,0.526
3,LGB_bal,0.661871,0.673043,0.640849,0.661871,0.0,0.520
4,XGB_bal,0.661539,0.677657,0.642186,0.661539,0.0,0.518


No model reached 0.80/0.80 in CV folds; this target may be infeasible without stronger signal/features.
Trained models: ['XGB_bal', 'LGB_bal', 'CAT_bal', 'LGB_tuned_best', 'STACK_blend', 'OOF_meta_cal', 'TwoStage_gate']
Train positive rate: 0.448 | Positive-majority mode: False
Targets -> accuracy>=0.80, recall>=0.80
GPU used -> XGB:True | LGBM:True | CAT:True


## 11) Threshold search on calibration split

In [48]:
def eval_at_threshold(y_true, p, thr):
    y_pred = (p >= thr).astype(int)
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    specificity = tn / max(tn + fp, 1)
    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'specificity': specificity,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    }

target_accuracy = globals().get('target_accuracy', 0.80)
target_recall = globals().get('target_recall', 0.80)

threshold_rows = []
for name, mdl in fitted_models.items():
    p_cal = mdl.predict_proba(Xcal)[:, 1]
    for thr in np.linspace(0.1, 0.9, 81):
        m = eval_at_threshold(y_cal, p_cal, thr)
        threshold_rows.append({
            'model': name, 'threshold': thr,
            **m,
            'dual_min': min(m['accuracy'], m['recall']),
            'meets_dual_target': int((m['accuracy'] >= target_accuracy) and (m['recall'] >= target_recall))
        })

threshold_df = pd.DataFrame(threshold_rows)
best_rows = []
for name, g in threshold_df.groupby('model'):
    feasible = g[(g['accuracy'] >= target_accuracy) & (g['recall'] >= target_recall)]
    pool = feasible if not feasible.empty else g
    top = pool.sort_values(['dual_min', 'recall', 'accuracy', 'f1', 'precision'], ascending=False).iloc[0].copy()
    top['selection_mode'] = 'dual_target_met' if not feasible.empty else 'fallback_best_dual_min'
    best_rows.append(top)

best_thresholds = pd.DataFrame(best_rows).reset_index(drop=True)
display(best_thresholds.sort_values(['meets_dual_target', 'dual_min', 'recall', 'accuracy'], ascending=False))
print(f'Dual target constraints: accuracy>={target_accuracy:.2f}, recall>={target_recall:.2f}')

,model,threshold,accuracy,recall,f1,precision,specificity,tp,tn,fp,fn,dual_min,meets_dual_target,selection_mode
2,LGB_tuned_best,0.53,0.667918,0.670528,0.644127,0.619727,0.665797,10650,13019,6535,5233,0.667918,0,fallback_best_dual_min
3,OOF_meta_cal,0.47,0.667636,0.668136,0.643113,0.619896,0.667229,10612,13047,6507,5271,0.667636,0,fallback_best_dual_min
4,STACK_blend,0.47,0.666253,0.679846,0.646142,0.615621,0.655211,10798,12812,6742,5085,0.666253,0,fallback_best_dual_min
0,CAT_bal,0.53,0.666196,0.666121,0.641426,0.618496,0.666258,10580,13028,6526,5303,0.666121,0,fallback_best_dual_min
5,TwoStage_gate,0.50,0.664475,0.666625,0.640416,0.616190,0.662729,10588,12959,6595,5295,0.664475,0,fallback_best_dual_min
1,LGB_bal,0.52,0.664221,0.674117,0.642812,0.614286,0.656183,10707,12831,6723,5176,0.664221,0,fallback_best_dual_min
6,XGB_bal,0.52,0.662923,0.672984,0.641539,0.612901,0.654751,10689,12803,6751,5194,0.662923,0,fallback_best_dual_min


Dual target constraints: accuracy>=0.80, recall>=0.80


## 12) Calibration (base, Platt, Isotonic, optional Venn-Abers)

In [22]:
try:
    from venn_abers import VennAbers
    venn_abers_available = True
except Exception:
    venn_abers_available = False

calib_rows = []
for name, mdl in fitted_models.items():
    p_cal_base = mdl.predict_proba(Xcal)[:, 1]
    p_te_base = mdl.predict_proba(Xte)[:, 1]

    platt = LogisticRegression(max_iter=2000)
    platt.fit(p_cal_base.reshape(-1, 1), y_cal)
    p_te_platt = platt.predict_proba(p_te_base.reshape(-1, 1))[:, 1]

    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(p_cal_base, y_cal)
    p_te_iso = iso.predict(p_te_base)

    preds = {
        'base': p_te_base,
        'platt': p_te_platt,
        'isotonic': p_te_iso
    }

    if venn_abers_available:
        va = VennAbers()
        fit2d = np.column_stack([1 - p_cal_base, p_cal_base])
        te2d = np.column_stack([1 - p_te_base, p_te_base])
        va.fit(fit2d, y_cal.values)
        p0, p1 = va.predict_proba(te2d)
        p1 = np.asarray(p1)
        preds['venn_abers'] = p1[:, 1] if p1.ndim == 2 else p1

    for method, p in preds.items():
        y_hat = (p >= 0.5).astype(int)
        calib_rows.append({
            'model': name,
            'calibration': method,
            'test_accuracy': accuracy_score(y_te, y_hat),
            'test_recall': recall_score(y_te, y_hat, zero_division=0),
            'test_f1': f1_score(y_te, y_hat, zero_division=0),
            'test_auc': roc_auc_score(y_te, p),
            'test_logloss': log_loss(y_te, p, labels=[0, 1]),
        })

calibration_results = pd.DataFrame(calib_rows)
calibration_results['test_min_acc_rec_f1'] = calibration_results[['test_accuracy', 'test_recall', 'test_f1']].min(axis=1)
display(calibration_results.sort_values('test_min_acc_rec_f1', ascending=False).head(15))

,model,calibration,test_accuracy,test_recall,test_f1,test_auc,test_logloss,test_min_acc_rec_f1
8,CAT_bal,base,0.662509,0.729224,0.659511,0.730485,0.605368,0.659511
12,LGB_optuna,base,0.662537,0.728658,0.659356,0.730037,0.605554,0.659356
4,LGB_bal,base,0.662650,0.715374,0.655287,0.728314,0.606382,0.655287
0,XGB_bal,base,0.662283,0.714367,0.654723,0.728627,0.606486,0.654723
23,OOF_meta_cal,venn_abers,0.666178,0.670738,0.643008,0.730084,0.605688,0.643008
22,OOF_meta_cal,isotonic,0.666149,0.670675,0.642967,0.730073,0.606561,0.642967
18,STACK_blend,isotonic,0.666573,0.669290,0.642784,0.730063,0.606475,0.642784
19,STACK_blend,venn_abers,0.666319,0.669731,0.642760,0.730208,0.605628,0.642760
9,CAT_bal,platt,0.666939,0.640204,0.632774,0.730485,0.601378,0.632774
16,STACK_blend,base,0.666432,0.640519,0.632534,0.730523,0.601338,0.632534


## 13) Final test evaluation with model-specific best threshold

In [49]:
target_accuracy = globals().get('target_accuracy', 0.80)
target_recall = globals().get('target_recall', 0.80)

final_rows = []
for _, r in best_thresholds.iterrows():
    name = r['model']
    thr = float(r['threshold'])
    p_te = fitted_models[name].predict_proba(Xte)[:, 1]
    m = eval_at_threshold(y_te, p_te, thr)
    final_rows.append({
        'model': name,
        'threshold': thr,
        **m,
        'auc': roc_auc_score(y_te, p_te),
        'dual_min': min(m['accuracy'], m['recall']),
        'meets_dual_target': int((m['accuracy'] >= target_accuracy) and (m['recall'] >= target_recall))
    })

final_test_results = pd.DataFrame(final_rows).sort_values(
    ['meets_dual_target', 'dual_min', 'recall', 'accuracy', 'f1', 'precision'], ascending=False
).reset_index(drop=True)

display(final_test_results)
print(f"Best row meets dual target: {bool(final_test_results.iloc[0]['meets_dual_target'])}")

,model,threshold,accuracy,recall,f1,precision,specificity,tp,tn,fp,fn,auc,dual_min,meets_dual_target
0,LGB_tuned_best,0.53,0.665867,0.678670,0.645490,0.615402,0.655467,10780,12817,6737,5104,0.728263,0.665867,0
1,OOF_meta_cal,0.47,0.665613,0.676467,0.644571,0.615548,0.656797,10745,12843,6711,5139,0.728290,0.665613,0
2,STACK_blend,0.47,0.665049,0.688806,0.648317,0.612324,0.645750,10941,12627,6927,4943,0.728313,0.665049,0
3,CAT_bal,0.53,0.664654,0.675334,0.643530,0.614587,0.655978,10727,12827,6727,5157,0.728065,0.664654,0
4,TwoStage_gate,0.50,0.664456,0.677726,0.644206,0.613845,0.653677,10765,12782,6772,5119,0.726046,0.664456,0
5,XGB_bal,0.52,0.664089,0.684651,0.646283,0.611986,0.647387,10875,12659,6895,5009,0.725382,0.664089,0
6,LGB_bal,0.52,0.663864,0.683077,0.645603,0.612026,0.648256,10850,12676,6878,5034,0.724927,0.663864,0


Best row meets dual target: False


## 14) Explainability (permutation importance)

In [50]:
best_model_name = final_test_results.iloc[0]['model']
best_model = fitted_models[best_model_name]

# permutation_importance requires sklearn estimator tags; custom OOF wrapper may not provide them.
if not hasattr(best_model, '__sklearn_tags__'):
    fallback_name = None
    for _, row in final_test_results.iterrows():
        nm = row['model']
        mdl = fitted_models[nm]
        if hasattr(mdl, '__sklearn_tags__'):
            fallback_name = nm
            best_model = mdl
            break
    print(f'Explainability fallback model: {fallback_name}')

perm = permutation_importance(
    best_model, Xte, y_te,
    scoring='f1', n_repeats=5, random_state=RANDOM_SEED, n_jobs=-1
)

importance_df = pd.DataFrame({
    'feature': Xte.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False).reset_index(drop=True)

display(importance_df.head(30))

,feature,importance_mean,importance_std
0,age,0.097139,0.002042
1,weight,0.009903,0.000832
2,vita,0.004969,0.000588
3,risk_metabolic_proxy,0.003052,0.000506
4,waist,0.002420,0.000358
5,miss_ind__drnk_30d_num,0.001897,0.001111
6,csc,0.001716,0.000200
7,int_ratio__chol__tri,0.001426,0.000406
8,fbs,0.001356,0.000183
9,int_prod__fbs__tri,0.000875,0.000196


## 15) Save artifacts and summary

In [51]:
out_dir = ROOT

final_test_results.to_csv(out_dir / 'clean_pipeline_final_test_results.csv', index=False)
calibration_results.to_csv(out_dir / 'clean_pipeline_calibration_results.csv', index=False)
importance_df.to_csv(out_dir / 'clean_pipeline_feature_importance.csv', index=False)

best_row = final_test_results.iloc[0]
summary_payload = {
    'best_model': str(best_row['model']),
    'best_threshold': float(best_row['threshold']),
    'best_dual_min': float(best_row['dual_min']),
    'best_accuracy': float(best_row['accuracy']),
    'best_recall': float(best_row['recall']),
    'best_f1': float(best_row['f1']),
    'best_precision': float(best_row['precision']),
    'best_specificity': float(best_row['specificity']),
    'meets_dual_target': bool(best_row['meets_dual_target']),
}

with open(out_dir / 'clean_pipeline_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary_payload, f, indent=2)

print('Saved: clean_pipeline_final_test_results.csv')
print('Saved: clean_pipeline_calibration_results.csv')
print('Saved: clean_pipeline_feature_importance.csv')
print('Saved: clean_pipeline_summary.json')
print('\nSummary:', summary_payload)

Saved: clean_pipeline_final_test_results.csv
Saved: clean_pipeline_calibration_results.csv
Saved: clean_pipeline_feature_importance.csv
Saved: clean_pipeline_summary.json

Summary: {'best_model': 'LGB_tuned_best', 'best_threshold': 0.53, 'best_dual_min': 0.6658671482589311, 'best_accuracy': 0.6658671482589311, 'best_recall': 0.6786703601108033, 'best_f1': 0.6454896559983234, 'best_precision': 0.615402180738711, 'best_specificity': 0.6554669121407385, 'meets_dual_target': False}


## 16) Extended benchmark: models + imputation + sampling (all requested techniques)

This section benchmarks all suggested options in one place:
- Models: XGBoost, LightGBM (GBDT + DART), CatBoost, BalancedRandomForest, EasyEnsemble
- Imputation: none (for NaN-tolerant trees), median, KNN(11), Iterative (MissForest-like), hybrid
- Sampling: none, SMOTEENN, BorderlineSMOTE, SMOTETomek
- Also includes CatBoost native-categorical branch (no one-hot).

In [33]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.ensemble import ExtraTreesRegressor
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.ensemble import EasyEnsembleClassifier
from imblearn.combine import SMOTEENN, SMOTETomek
from imblearn.over_sampling import BorderlineSMOTE

# -------- Utility metrics / threshold --------
def _metrics_at_thr(y_true, p, thr):
    y_hat = (p >= thr).astype(int)
    acc = accuracy_score(y_true, y_hat)
    rec = recall_score(y_true, y_hat, zero_division=0)
    f1 = f1_score(y_true, y_hat, zero_division=0)
    pre = precision_score(y_true, y_hat, zero_division=0)
    auc = roc_auc_score(y_true, p)
    tp = int(((y_true == 1) & (y_hat == 1)).sum())
    tn = int(((y_true == 0) & (y_hat == 0)).sum())
    fp = int(((y_true == 0) & (y_hat == 1)).sum())
    fn = int(((y_true == 1) & (y_hat == 0)).sum())
    specificity = tn / max(tn + fp, 1)
    return {
        'accuracy': acc,
        'recall': rec,
        'f1': f1,
        'precision': pre,
        'specificity': specificity,
        'auc': auc,
        'min_metric': min(acc, rec, f1),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
    }


def _best_threshold(y_true, p, min_precision=0.60, min_specificity=0.30):
    grid = np.linspace(0.10, 0.90, 81)
    rows = []
    for t in grid:
        m = _metrics_at_thr(y_true, p, t)
        rows.append({'threshold': t, **m})
    sweep = pd.DataFrame(rows)

    feasible = sweep[(sweep['precision'] >= min_precision) & (sweep['specificity'] >= min_specificity)]
    pool = feasible if not feasible.empty else sweep
    best = pool.sort_values(['recall', 'f1', 'precision', 'specificity', 'accuracy'], ascending=False).iloc[0]
    return float(best['threshold']), sweep


# -------- Imputation strategies --------
def _impute_triplet(Xtr_in, Xcal_in, Xte_in, strategy='none'):
    Xtr0, Xcal0, Xte0 = Xtr_in.copy(), Xcal_in.copy(), Xte_in.copy()

    if strategy == 'none':
        return Xtr0, Xcal0, Xte0

    if strategy == 'median':
        med = Xtr0.median(numeric_only=True)
        return Xtr0.fillna(med), Xcal0.fillna(med), Xte0.fillna(med)

    if strategy == 'knn11':
        imp = KNNImputer(n_neighbors=11)
        cols = Xtr0.columns
        Xtr1 = pd.DataFrame(imp.fit_transform(Xtr0), columns=cols, index=Xtr0.index)
        Xcal1 = pd.DataFrame(imp.transform(Xcal0), columns=cols, index=Xcal0.index)
        Xte1 = pd.DataFrame(imp.transform(Xte0), columns=cols, index=Xte0.index)
        return Xtr1, Xcal1, Xte1

    if strategy == 'iterative_missforest':
        imp = IterativeImputer(
            estimator=ExtraTreesRegressor(n_estimators=40, max_depth=12, random_state=RANDOM_SEED, n_jobs=1),
            max_iter=5,
            initial_strategy='median',
            skip_complete=True,
            random_state=RANDOM_SEED,
        )
        cols = Xtr0.columns
        Xtr1 = pd.DataFrame(imp.fit_transform(Xtr0), columns=cols, index=Xtr0.index)
        Xcal1 = pd.DataFrame(imp.transform(Xcal0), columns=cols, index=Xcal0.index)
        Xte1 = pd.DataFrame(imp.transform(Xte0), columns=cols, index=Xte0.index)
        return Xtr1, Xcal1, Xte1

    if strategy == 'hybrid':
        # Hybrid: KNN on moderate-missing columns, median elsewhere
        miss = Xtr0.isna().mean()
        knn_cols = miss[(miss > 0.02) & (miss <= 0.35)].index.tolist()
        med_cols = [c for c in Xtr0.columns if c not in knn_cols]

        Xtr1, Xcal1, Xte1 = Xtr0.copy(), Xcal0.copy(), Xte0.copy()

        if med_cols:
            med = Xtr1[med_cols].median(numeric_only=True)
            Xtr1[med_cols] = Xtr1[med_cols].fillna(med)
            Xcal1[med_cols] = Xcal1[med_cols].fillna(med)
            Xte1[med_cols] = Xte1[med_cols].fillna(med)

        if knn_cols:
            imp = KNNImputer(n_neighbors=11)
            Xtr_knn = pd.DataFrame(imp.fit_transform(Xtr1[knn_cols]), columns=knn_cols, index=Xtr1.index)
            Xcal_knn = pd.DataFrame(imp.transform(Xcal1[knn_cols]), columns=knn_cols, index=Xcal1.index)
            Xte_knn = pd.DataFrame(imp.transform(Xte1[knn_cols]), columns=knn_cols, index=Xte1.index)
            Xtr1[knn_cols] = Xtr_knn
            Xcal1[knn_cols] = Xcal_knn
            Xte1[knn_cols] = Xte_knn

        # safety fill
        med_all = Xtr1.median(numeric_only=True)
        Xtr1 = Xtr1.fillna(med_all)
        Xcal1 = Xcal1.fillna(med_all)
        Xte1 = Xte1.fillna(med_all)
        return Xtr1, Xcal1, Xte1

    raise ValueError(f'Unknown imputation strategy: {strategy}')


# -------- Sampling --------
def _sample_train(Xtr_in, ytr_in, sampling='none'):
    if sampling == 'none':
        return Xtr_in, ytr_in
    if sampling == 'smoteenn':
        sampler = SMOTEENN(random_state=RANDOM_SEED)
    elif sampling == 'borderline_smote':
        sampler = BorderlineSMOTE(random_state=RANDOM_SEED)
    elif sampling == 'smotetomek':
        sampler = SMOTETomek(random_state=RANDOM_SEED)
    else:
        raise ValueError(f'Unknown sampling strategy: {sampling}')

    Xs, ys = sampler.fit_resample(Xtr_in, ytr_in)
    Xs = pd.DataFrame(Xs, columns=Xtr_in.columns)
    ys = pd.Series(ys)
    return Xs, ys


# -------- Model factory --------
def _make_model(name, ytr_ref):
    pos_rate = float(ytr_ref.mean())
    pos_majority = pos_rate >= 0.60
    spw = (ytr_ref.eq(0).sum() / max(ytr_ref.eq(1).sum(), 1))
    if pos_majority:
        spw = 1.0

    xgb_gpu_params = {'device': 'cuda', 'tree_method': 'hist'} if XGB_GPU_AVAILABLE else {'tree_method': 'hist'}
    lgbm_gpu_params = {'device_type': 'gpu'} if LGBM_GPU_AVAILABLE else {}
    cat_gpu_params = {'task_type': 'GPU'} if CAT_GPU_AVAILABLE else {}

    if name == 'xgb_bal':
        return XGBClassifier(
            n_estimators=500, learning_rate=0.05, max_depth=6,
            subsample=0.8, colsample_bytree=0.8,
            scale_pos_weight=spw, eval_metric='logloss',
            random_state=RANDOM_SEED, verbosity=0,
            **xgb_gpu_params
        )

    if name == 'lgbm_bal':
        return LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8,
            class_weight=None if pos_majority else 'balanced', random_state=RANDOM_SEED, verbose=-1,
            **lgbm_gpu_params
        )

    if name == 'lgbm_dart':
        return LGBMClassifier(
            boosting_type='dart',
            n_estimators=600, learning_rate=0.04, num_leaves=63,
            subsample=0.8, colsample_bytree=0.8,
            class_weight=None if pos_majority else 'balanced', random_state=RANDOM_SEED, verbose=-1,
            **lgbm_gpu_params
        )

    if name == 'cat_bal':
        cat_weight_params = {} if pos_majority else {'auto_class_weights': 'Balanced'}
        return CatBoostClassifier(
            iterations=500, learning_rate=0.05, depth=6,
            random_seed=RANDOM_SEED, verbose=0,
            **cat_weight_params,
            **cat_gpu_params
        )

    if name == 'balanced_rf':
        return BalancedRandomForestClassifier(
            n_estimators=500, random_state=RANDOM_SEED, n_jobs=-1
        )

    if name == 'easy_ensemble':
        return EasyEnsembleClassifier(
            n_estimators=20, random_state=RANDOM_SEED, n_jobs=-1
        )

    raise ValueError(f'Unknown model: {name}')


# -------- Run matrix-based experiment --------
def _run_matrix_experiment(exp_name, model_name, impute='none', sampling='none'):
    # Start from one-hot encoded split from earlier notebook cells
    Xtr0, Xcal0, Xte0 = X_tr.copy(), X_cal.copy(), X_te.copy()
    ytr0, ycal0, yte0 = y_tr.copy(), y_cal.copy(), y_te.copy()

    # Imputation
    Xtr1, Xcal1, Xte1 = _impute_triplet(Xtr0, Xcal0, Xte0, strategy=impute)

    # Sampling only on train
    Xtr2, ytr2 = _sample_train(Xtr1, ytr0, sampling=sampling)

    # Scale (helps some models, safe for all here)
    scaler_b = StandardScaler()
    Xtr2s = pd.DataFrame(scaler_b.fit_transform(Xtr2), columns=Xtr2.columns, index=Xtr2.index)
    Xcal1s = pd.DataFrame(scaler_b.transform(Xcal1), columns=Xcal1.columns, index=Xcal1.index)
    Xte1s = pd.DataFrame(scaler_b.transform(Xte1), columns=Xte1.columns, index=Xte1.index)

    # Fit / threshold / test
    mdl = _make_model(model_name, ytr2)
    mdl.fit(Xtr2s, ytr2)

    p_cal = mdl.predict_proba(Xcal1s)[:, 1]
    min_precision = 0.60 if float(ytr2.mean()) >= 0.60 else 0.50
    min_specificity = 0.30 if float(ytr2.mean()) >= 0.60 else 0.20
    thr, _ = _best_threshold(ycal0, p_cal, min_precision=min_precision, min_specificity=min_specificity)

    p_te = mdl.predict_proba(Xte1s)[:, 1]
    m = _metrics_at_thr(yte0, p_te, thr)

    return {
        'experiment': exp_name,
        'model': model_name,
        'imputation': impute,
        'sampling': sampling,
        'threshold': thr,
        'test_accuracy': m['accuracy'],
        'test_recall': m['recall'],
        'test_f1': m['f1'],
        'test_precision': m['precision'],
        'test_specificity': m['specificity'],
        'test_auc': m['auc'],
        'test_min_acc_rec_f1': m['min_metric'],
    }


# -------- CatBoost native-categorical experiment --------
def _run_catboost_native(exp_name='catboost_native_categorical'):
    # Rebuild raw (non-one-hot) matrices aligned to the same split indices
    X_raw = merged_data.drop(columns=[TARGET_COL]).copy()

    Xtr_raw = X_raw.loc[y_tr.index].copy()
    Xcal_raw = X_raw.loc[y_cal.index].copy()
    Xte_raw = X_raw.loc[y_te.index].copy()

    # Identify categorical by dtype; keep as string with explicit missing token
    cat_cols = Xtr_raw.select_dtypes(include=['object', 'category']).columns.tolist()
    for c in cat_cols:
        Xtr_raw[c] = Xtr_raw[c].astype(str).fillna('__MISSING__')
        Xcal_raw[c] = Xcal_raw[c].astype(str).fillna('__MISSING__')
        Xte_raw[c] = Xte_raw[c].astype(str).fillna('__MISSING__')

    # Numeric safety fill for non-cat
    num_cols = [c for c in Xtr_raw.columns if c not in cat_cols]
    med = Xtr_raw[num_cols].median(numeric_only=True)
    Xtr_raw[num_cols] = Xtr_raw[num_cols].fillna(med)
    Xcal_raw[num_cols] = Xcal_raw[num_cols].fillna(med)
    Xte_raw[num_cols] = Xte_raw[num_cols].fillna(med)

    cat_idx = [Xtr_raw.columns.get_loc(c) for c in cat_cols]
    cat_gpu_params = {'task_type': 'GPU'} if CAT_GPU_AVAILABLE else {}
    cat_weight_params = {} if float(y_tr.mean()) >= 0.60 else {'auto_class_weights': 'Balanced'}

    mdl = CatBoostClassifier(
        iterations=600, learning_rate=0.04, depth=6,
        random_seed=RANDOM_SEED, verbose=0,
        **cat_weight_params,
        **cat_gpu_params
    )
    mdl.fit(Xtr_raw, y_tr, cat_features=cat_idx)

    p_cal = mdl.predict_proba(Xcal_raw)[:, 1]
    min_precision = 0.60 if float(y_tr.mean()) >= 0.60 else 0.50
    min_specificity = 0.30 if float(y_tr.mean()) >= 0.60 else 0.20
    thr, _ = _best_threshold(y_cal, p_cal, min_precision=min_precision, min_specificity=min_specificity)
    p_te = mdl.predict_proba(Xte_raw)[:, 1]
    m = _metrics_at_thr(y_te, p_te, thr)

    return {
        'experiment': exp_name,
        'model': 'cat_bal_native_cat',
        'imputation': 'native_cat+median_num',
        'sampling': 'none',
        'threshold': thr,
        'test_accuracy': m['accuracy'],
        'test_recall': m['recall'],
        'test_f1': m['f1'],
        'test_precision': m['precision'],
        'test_specificity': m['specificity'],
        'test_auc': m['auc'],
        'test_min_acc_rec_f1': m['min_metric'],
    }

print('Benchmark helpers loaded.')
print(f"GPU flags -> XGB:{XGB_GPU_AVAILABLE} | LGBM:{LGBM_GPU_AVAILABLE} | CAT:{CAT_GPU_AVAILABLE}")

Benchmark helpers loaded.
GPU flags -> XGB:True | LGBM:True | CAT:True


In [34]:
train_pos_rate = float(y_tr.mean())
pos_majority_mode = train_pos_rate >= 0.60
print(f'Train positive rate: {train_pos_rate:.3f} | Positive-majority mode: {pos_majority_mode}')

# All requested experiments
benchmark_configs = [
    # NaN-tolerant tree models (no imputation)
    {'experiment': 'xgb_no_impute',              'model': 'xgb_bal',       'impute': 'none',                'sampling': 'none'},
    {'experiment': 'lgbm_no_impute',             'model': 'lgbm_bal',      'impute': 'none',                'sampling': 'none'},
    {'experiment': 'lgbm_dart_no_impute',        'model': 'lgbm_dart',     'impute': 'none',                'sampling': 'none'},
    {'experiment': 'cat_no_impute',              'model': 'cat_bal',       'impute': 'none',                'sampling': 'none'},

    # Imputation alternatives
    {'experiment': 'lgbm_median',                'model': 'lgbm_bal',      'impute': 'median',              'sampling': 'none'},
    {'experiment': 'lgbm_knn11',                 'model': 'lgbm_bal',      'impute': 'knn11',               'sampling': 'none'},
    {'experiment': 'lgbm_iterative_missforest',  'model': 'lgbm_bal',      'impute': 'iterative_missforest','sampling': 'none'},
    {'experiment': 'lgbm_hybrid',                'model': 'lgbm_bal',      'impute': 'hybrid',              'sampling': 'none'},

    # Additional ensemble baselines for imbalance
    {'experiment': 'balanced_rf_median',         'model': 'balanced_rf',   'impute': 'median',              'sampling': 'none'},
    {'experiment': 'easy_ensemble_median',       'model': 'easy_ensemble', 'impute': 'median',              'sampling': 'none'},
]

if not pos_majority_mode:
    # Only try positive oversampling when positive class is not already majority.
    benchmark_configs.extend([
        {'experiment': 'lgbm_knn11_smoteenn',   'model': 'lgbm_bal',      'impute': 'knn11',               'sampling': 'smoteenn'},
        {'experiment': 'lgbm_knn11_borderline', 'model': 'lgbm_bal',      'impute': 'knn11',               'sampling': 'borderline_smote'},
        {'experiment': 'lgbm_knn11_smotetomek', 'model': 'lgbm_bal',      'impute': 'knn11',               'sampling': 'smotetomek'},
    ])
else:
    print('Skipping SMOTE-style experiments because positives are already majority.')

progress_file = 'benchmark_progress.csv'
benchmark_rows = load_ckpt('16_benchmark_rows', default=[])
if not isinstance(benchmark_rows, list):
    benchmark_rows = []

done = {r.get('experiment') for r in benchmark_rows if isinstance(r, dict)}
print(f'Resuming benchmark. Already completed: {len(done)} experiments')

for cfg in benchmark_configs:
    exp = cfg['experiment']
    if exp in done:
        print(f'Skipping {exp} (already checkpointed)')
        continue

    print(f"Running {exp} ...")
    try:
        row = _run_matrix_experiment(
            exp_name=cfg['experiment'],
            model_name=cfg['model'],
            impute=cfg['impute'],
            sampling=cfg['sampling']
        )
        benchmark_rows.append(row)
        save_ckpt('16_benchmark_rows', benchmark_rows)
        save_df_progress(pd.DataFrame(benchmark_rows), progress_file)
        mark_stage(f'16_done_{exp}')
        done.add(exp)
    except MemoryError as e:
        print(f'Failed {exp} due to memory pressure: {e}')
        benchmark_rows.append({'experiment': exp, 'status': 'failed_memory_error'})
        save_ckpt('16_benchmark_rows', benchmark_rows)
        save_df_progress(pd.DataFrame(benchmark_rows), progress_file)
        done.add(exp)
        continue
    except Exception as e:
        print(f'Failed {exp}: {e}')
        save_ckpt('16_benchmark_rows', benchmark_rows)
        save_df_progress(pd.DataFrame(benchmark_rows), progress_file)
        raise

if 'catboost_native_categorical' not in done:
    print('Running catboost_native_categorical ...')
    row = _run_catboost_native()
    benchmark_rows.append(row)
    save_ckpt('16_benchmark_rows', benchmark_rows)
    save_df_progress(pd.DataFrame(benchmark_rows), progress_file)
    mark_stage('16_done_catboost_native')

benchmark_results = pd.DataFrame(benchmark_rows).sort_values('test_recall', ascending=False).reset_index(drop=True)

print('\nTop benchmark rows:')
display(benchmark_results.head(20))

benchmark_results.to_csv(ROOT / 'benchmark_all_methods.csv', index=False)
save_ckpt('16_benchmark_final', benchmark_results)
mark_stage('16_benchmark_done')
print('Saved: benchmark_all_methods.csv')

print('\nBest experiment:')
print(benchmark_results.iloc[0].to_string())

Train positive rate: 0.448 | Positive-majority mode: False
Resuming benchmark. Already completed: 14 experiments
Skipping xgb_no_impute (already checkpointed)
Skipping lgbm_no_impute (already checkpointed)
Skipping lgbm_dart_no_impute (already checkpointed)
Skipping cat_no_impute (already checkpointed)
Skipping lgbm_median (already checkpointed)
Skipping lgbm_knn11 (already checkpointed)
Skipping lgbm_iterative_missforest (already checkpointed)
Skipping lgbm_hybrid (already checkpointed)
Skipping balanced_rf_median (already checkpointed)
Skipping easy_ensemble_median (already checkpointed)
Skipping lgbm_knn11_smoteenn (already checkpointed)
Skipping lgbm_knn11_borderline (already checkpointed)
Skipping lgbm_knn11_smotetomek (already checkpointed)

Top benchmark rows:


,experiment,model,imputation,sampling,threshold,test_accuracy,test_recall,test_f1,test_precision,test_auc,test_min_acc_rec_f1
0,lgbm_knn11_borderline,lgbm_bal,knn11,borderline_smote,0.44,0.657825,0.747482,0.661965,0.594006,0.726560,0.657825
1,lgbm_iterative_missforest,lgbm_bal,iterative_missforest,none,0.48,0.659716,0.747041,0.663072,0.596072,0.726962,0.659716
2,lgbm_knn11_smotetomek,lgbm_bal,knn11,smotetomek,0.43,0.656414,0.746726,0.660817,0.592635,0.726048,0.656414
3,xgb_no_impute,xgb_bal,none,none,0.48,0.657712,0.746160,0.661495,0.594085,0.729636,0.657712
4,lgbm_no_impute,lgbm_bal,none,none,0.48,0.657571,0.745656,0.661251,0.594012,0.728983,0.657571
5,lgbm_dart_no_impute,lgbm_dart,none,none,0.49,0.659716,0.745593,0.662638,0.596294,0.729416,0.659716
6,lgbm_knn11,lgbm_bal,knn11,none,0.48,0.657063,0.744901,0.660692,0.593589,0.726955,0.657063
7,lgbm_median,lgbm_bal,median,none,0.48,0.658276,0.742256,0.660689,0.595274,0.727610,0.658276
8,catboost_native_categorical,cat_bal_native_cat,native_cat+median_num,none,0.49,0.660111,0.742067,0.661838,0.597264,0.730046,0.660111
9,cat_no_impute,cat_bal,none,none,0.49,0.660675,0.741627,0.662077,0.597939,0.730673,0.660675


Saved: benchmark_all_methods.csv

Best experiment:
experiment             lgbm_knn11_borderline
model                               lgbm_bal
imputation                             knn11
sampling                    borderline_smote
threshold                               0.44
test_accuracy                       0.657825
test_recall                         0.747482
test_f1                             0.661965
test_precision                      0.594006
test_auc                             0.72656
test_min_acc_rec_f1                 0.657825
